In [1]:
import pandas as pd
import sys
import os
import time
import joblib
import numpy as np

from numba import njit, prange

sys.path.insert(0, '../../..') # Load Parent directory to import utils
from utils.constants import Configs
from utils.helper_functions import normalize_series # calcula a distância entre séries temporais usando DTW

/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


### Load Data From cache

In [2]:
cache_path = Configs.SMART_RETAIL_FILTERED_DATA_BIMANUAL
if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Processed data not found at {cache_path}. Please run the data processing notebook first.")
df = pd.read_csv(cache_path)

### Delete Subject 13

In [3]:
# df.drop(df[df['sub']== 13].index, inplace=True)
# df[df['sub']==13].head()

### Calculate DTW Matrix for all movements
Em cada instante t, o ponto passa a ser:

x(t) = [X_L, Y_L, Z_L, X_R, Y_R, Z_R]

A série de um movimento é uma matriz (T × 6). O DTW dependente alinha as duas séries com um único caminho, e o custo local é a distância euclidiana em 6D:

d(i, j) = sqrt( Σ_k (a_ik − b_jk)² ), k = 1…6

Como o caminho é partilhado, os dois braços são obrigados a estar alinhados em simultâneo. Um movimento em que os braços estão dessincronizados em relação ao normal paga custo nas duas partes ao mesmo tempo. É isto que preserva a coordenação.

In [ ]:
from tslearn.metrics import cdist_dtw


group_both = ['x_L', 'y_L', 'z_L', 'x_R', 'y_R', 'z_R']

sample_size = None
window_percentages = [0.05, 0.10, 0.15, 0.20]

constrainted_times = []

for movement in ["Pick", "Park", "Place"]:
    print(f"\n--- Tuning Constrained Bimanual DTW for {movement} ---")
    
    normalized_both = normalize_series(
        df_input=df,
        movLabel_filter=movement,
        useScaler=False,
        group_array=group_both,
        sample_size=sample_size
    )
    seq_length = normalized_both.shape[1]
    
    best_score, best_radius, best_dtw_matrix, best_time = -1, None, None, 0.0
    for w_pct in window_percentages:
        radius = max(1, int(seq_length * w_pct))

        start_time = time.perf_counter()
        dtw_matrix = cdist_dtw(
            normalized_both,
            global_constraint="sakoe_chiba",
            sakoe_chiba_radius=radius,
            n_jobs=-1,
            be="pytorch"
        )
        elapsed_time = time.perf_counter() - start_time

        
        # Evaluate the separation score by calculating the variance of the upper triangular part of the fused DTW matrix
        upper_tri = dtw_matrix[np.triu_indices_from(dtw_matrix, k=1)]
        separation_score = np.var(upper_tri)
        
        print(f"  -> Radius: {radius} ({w_pct*100:.0f}%) | Time: {elapsed_time:.2f}s | Separation Score: {separation_score:.2f}")
        
        if separation_score > best_score:
            best_score, best_radius = separation_score, radius
            best_dtw_matrix, best_time = dtw_matrix, elapsed_time

    constrainted_times.append(best_time)
    
    print(f"[*] Selected optimal Sakoe-Chiba radius for {movement}: {best_radius}")
    print(f"[*] Final Representation Time: {best_time:.4f}s")
    
    # Export best matrix
    cache_name = str(Configs.dtw_matrix_6d_path(movement))
    joblib.dump(best_dtw_matrix, cache_name)
    print(f"[*] Saved optimal fused matrix to {cache_name}")


--- Tuning Constrained Bimanual DTW for Pick ---


Processing time series: 100%|██████████| 380/380 [00:00<00:00, 2279.80it/s]

Shape of first series: (106, 6)


Shape of formatted series: (380, 461, 6)
  -> Radius: 23 (5%) | Time: 126.16s | Separation Score: 3677810.19
  -> Radius: 46 (10%) | Time: 117.73s | Separation Score: 3677099.64
  -> Radius: 69 (15%) | Time: 110.43s | Separation Score: 3676680.71
  -> Radius: 92 (20%) | Time: 112.28s | Separation Score: 3676428.28
[*] Selected optimal Sakoe-Chiba radius for Pick: 23
[*] Final Representation Time: 126.1598s
[*] Saved optimal fused matrix to /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/SmartRetail/SmartRetail_DTW/cache/metadata/dtw_6d_distance_matrix_Pick.pkl

--- Tuning Constrained Bimanual DTW for Park ---


Processing time series: 100%|██████████| 185/185 [00:00<00:00, 2891.06it/s]

Shape of first series: (46, 6)


Shape of formatted series: (185, 466, 6)
  -> Radius: 23 (5%) | Time: 26.00s | Separation Score: 1641030.63
  -> Radius: 46 (10%) | Time: 25.73s | Separation Score: 1639671.49
  -> Radius: 69 (15%) | Time: 26.11s | Separation Score: 1639108.87
  -> Radius: 93 (20%) | Time: 27.24s | Separation Score: 1638844.86
[*] Selected optimal Sakoe-Chiba radius for Park: 23
[*] Final Representation Time: 25.9992s
[*] Saved optimal fused matrix to /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/SmartRetail/SmartRetail_DTW/cache/metadata/dtw_6d_distance_matrix_Park.pkl

--- Tuning Constrained Bimanual DTW for Place ---


Processing time series: 100%|██████████| 380/380 [00:00<00:00, 2513.39it/s]

Shape of first series: (121, 6)


Shape of formatted series: (380, 2126, 6)
  -> Radius: 106 (5%) | Time: 528.47s | Separation Score: 6795628.52
  -> Radius: 212 (10%) | Time: 586.31s | Separation Score: 6793680.37
  -> Radius: 318 (15%) | Time: 613.09s | Separation Score: 6793679.06
  -> Radius: 425 (20%) | Time: 512.62s | Separation Score: 6793679.06
[*] Selected optimal Sakoe-Chiba radius for Place: 106
[*] Final Representation Time: 528.4724s
[*] Saved optimal fused matrix to /home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/SmartRetail/SmartRetail_DTW/cache/metadata/dtw_6d_distance_matrix_Place.pkl


In [5]:
print("\nFinal times per phase [Pick, Park, Place]:", [round(t, 2) for t in constrainted_times])


Final times per phase [Pick, Park, Place]: [126.16, 26.0, 528.47]
